## Importar Librerías

In [1]:
import json
import os
import pandas as pd
import numpy as np

Configuración

In [2]:
PATH_JSON = "logs_KX7A_20260220-1944_GrupoB.json"

# Formato típico de Moodle (ejemplo: "20/02/26, 19:44:01")
FORMATO_HORA = "%d/%m/%y, %H:%M:%S"

# Ventana nocturna (incluye 0 a 5)
NIGHT_START = 0
NIGHT_END = 5

# Percentil para definir "picos" de actividad diaria (0.90 = top 10%)
PICO_Q = 0.90

# Columnas esperadas en TU df (según tu screenshot)
COL_TIME = "hora"
COL_ACTOR = "nombrecompletodelusuario"

## UTILIDADES

In [3]:

def assert_file_exists(path: str) -> None:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No se encontró el archivo: {path}\n"
            f"Contenido de /content: {os.listdir('/content') if os.path.exists('/content') else 'N/A'}\n"
            "Solución: sube el JSON al panel Files de Colab, o ajusta PATH_JSON."
        )


def load_moodle_json(path: str) -> list:
    """Carga JSON y corrige el anidamiento doble si existe."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Arreglar anidamiento doble: [ [ {...}, {...} ] ]
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
        return data[0]
    return data


def normalize_records(records: list) -> pd.DataFrame:
    """Convierte lista de dicts a DataFrame."""
    df = pd.json_normalize(records)

    # Validación mínima de columnas
    missing = [c for c in [COL_TIME, COL_ACTOR] if c not in df.columns]
    if missing:
        raise KeyError(
            f"Faltan columnas esperadas: {missing}\n"
            f"Columnas disponibles: {list(df.columns)}\n"
            "Solución: revisa df.columns y ajusta COL_TIME / COL_ACTOR."
        )

    return df


def parse_datetime(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte la columna de tiempo a datetime y crea variables temporales."""
    df = df.copy()

    df[COL_TIME] = pd.to_datetime(
        df[COL_TIME],
        format=FORMATO_HORA,
        errors="coerce"
    )

    # Eliminar filas con fecha inválida
    df = df.dropna(subset=[COL_TIME])

    df["semana"] = df[COL_TIME].dt.isocalendar().week.astype(int)
    df["hora_num"] = df[COL_TIME].dt.hour.astype(int)
    df["fecha"] = df[COL_TIME].dt.date

    return df

## CÁLCULO PATRÓN TEMPORAL

In [4]:
def calcular_patron_temporal(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula variables de patrón temporal por actor.
    Retorna un DF con una fila por estudiante.
    """
    resultados = []

    for actor, grupo in df.groupby(COL_ACTOR):
        # --- 1) Actividad semanal (estabilidad) ---
        actividad_semanal = grupo.groupby("semana").size()
        media_semanal = actividad_semanal.mean()
        desviacion_semanal = actividad_semanal.std(ddof=1)  # std muestral

        cv_semanal = (desviacion_semanal / media_semanal) if media_semanal and media_semanal > 0 else 0.0

        # --- 2) Actividad nocturna ---
        eventos_nocturnos = grupo[(grupo["hora_num"] >= NIGHT_START) & (grupo["hora_num"] <= NIGHT_END)]
        proporcion_nocturna = len(eventos_nocturnos) / len(grupo) if len(grupo) > 0 else 0.0

        # --- 3) Intensidad de picos ---
        actividad_diaria = grupo.groupby("fecha").size()

        if len(actividad_diaria) > 0:
            umbral_pico = actividad_diaria.quantile(PICO_Q)
            dias_pico = actividad_diaria[actividad_diaria >= umbral_pico]
            intensidad_picos = dias_pico.sum() / actividad_diaria.sum()
        else:
            intensidad_picos = 0.0

        # --- 4) Rigidez horaria ---
        # (std baja = patrón muy fijo; std alta = dispersión)
        desviacion_horaria = float(grupo["hora_num"].std(ddof=1)) if len(grupo) > 1 else 0.0

        resultados.append({
            "actor": actor,
            "eventos_total": int(len(grupo)),
            "semanas_con_actividad": int(actividad_semanal.shape[0]),
            "cv_semanal": float(cv_semanal),
            "proporcion_nocturna": float(proporcion_nocturna),
            "intensidad_picos": float(intensidad_picos),
            "desviacion_horaria": float(desviacion_horaria)
        })

    return pd.DataFrame(resultados).sort_values("actor").reset_index(drop=True)

## EJECUCIÓN

In [5]:
def main():
    assert_file_exists(PATH_JSON)

    records = load_moodle_json(PATH_JSON)
    df_raw = normalize_records(records)

    print("Columnas detectadas:")
    print(list(df_raw.columns))

    df = parse_datetime(df_raw)

    print("\nPreview (df procesado):")
    display(df.head())

    patron_temporal_df = calcular_patron_temporal(df)

    print("\nPreview (patron_temporal_df):")
    display(patron_temporal_df.head())

    # Guardar a CSV para descargar o usar en Power BI
    out_csv = "patron_temporal_resultados.csv"
    patron_temporal_df.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"\nArchivo generado: {out_csv}")

    return patron_temporal_df

In [6]:
# Ejecutar
patron_temporal_df = main()

FileNotFoundError: No se encontró el archivo: logs_KX7A_20260220-1944_GrupoB.json
Contenido de /content: ['.config', 'sample_data']
Solución: sube el JSON al panel Files de Colab, o ajusta PATH_JSON.

In [ ]:
# (Optional, only if you get an engine error in Colab)
!pip -q install openpyxl

# Export results
patron_temporal_df.to_excel("patron_temporal_resultados.xlsx", index=False)

## Alt - V2

## Importar / Dar formato

In [ ]:
import json
import pandas as pd
import numpy as np

path = "/content/logs_KX7A_20260220-1944_GrupoB.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

## Anidamiento
if isinstance(data, list) and len(data) == 1 and isinstance(data[0], list):
    records = data[0]
else:
    records = data

df = pd.json_normalize(records)
df.head()

## Arreglar formato de fecha

df["hora"] = pd.to_datetime(
    df["hora"],
    format="%d/%m/%y, %H:%M:%S",
    errors="coerce"
)

df["fecha"] = df["hora"].dt.date

events_por_dia = (
    df.groupby("fecha")
      .size()
      .reset_index(name="eventos")
      .sort_values("fecha")
)

events_por_dia.head()

# Asegurar que la columna de tiempo sea tipo datetime
df["event_time"] = pd.to_datetime(df["event_time"])

# Extraer variables temporales
df["semana"] = df["event_time"].dt.isocalendar().week
df["hora"] = df["event_time"].dt.hour
df["fecha"] = df["event_time"].dt.date

resultados = []

In [ ]:
df.columns

In [ ]:
# Agrupar por estudiante (actor)
for actor, grupo in df.groupby("actor_name"):

    # ---- 1. Actividad semanal ----
    actividad_semanal = grupo.groupby("semana").size()
    media_semanal = actividad_semanal.mean()
    desviacion_semanal = actividad_semanal.std()

    # Coeficiente de variación (estabilidad de rutina)
    cv = desviacion_semanal / media_semanal if media_semanal > 0 else 0

    # ---- 2. Actividad nocturna (00:00 - 05:59) ----
    eventos_nocturnos = grupo[(grupo["hora"] >= 0) & (grupo["hora"] <= 5)]
    proporcion_nocturna = len(eventos_nocturnos) / len(grupo)

    # ---- 3. Intensidad de picos ----
    actividad_diaria = grupo.groupby("fecha").size()
    umbral_pico = actividad_diaria.quantile(0.9)  # Percentil 90
    dias_pico = actividad_diaria[actividad_diaria >= umbral_pico]
    intensidad_picos = dias_pico.sum() / actividad_diaria.sum()

    # ---- 4. Rigidez horaria ----
    desviacion_horaria = grupo["hora"].std()

    resultados.append({
        "actor_name": actor,
        "cv_semanal": cv,
        "proporcion_nocturna": proporcion_nocturna,
        "intensidad_picos": intensidad_picos,
        "desviacion_horaria": desviacion_horaria
    })

# DataFrame final con variables del patrón temporal
patron_temporal_df = pd.DataFrame(resultados)